# M2.5 · Train/serve skew & the feature contract

_Curriculum · Domain 0 · ML Foundations · Feature engineering & leakage_

_Save a copy to your Drive_

A feature contract says training and serving must compute the same feature the same way for the same request. In this notebook we intentionally break that contract with a 7-day offline window and a 1-day online window, then repair it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(25)

## Build deterministic campaign histories

Each row is one served ad opportunity. The true click probability depends on a 7-day campaign click count, so training on the 7-day feature is coherent.

In [ ]:
n = 1200

campaign_quality = rng.gamma(shape=2.0, scale=12.0, size=n)
clicks_7d = rng.poisson(lam=campaign_quality + 8.0)
share_recent = rng.beta(a=2.0, b=5.0, size=n)
clicks_1d = rng.binomial(n=clicks_7d, p=share_recent)

logits = -3.2 + 0.055 * clicks_7d
prob = 1.0 / (1.0 + np.exp(-logits))
clicked = (rng.random(n) < prob).astype(int)

df = pd.DataFrame({
    "clicks_7d": clicks_7d,
    "clicks_1d": clicks_1d,
    "clicked": clicked
})

df.head()

## Fit a tiny logistic model on the offline contract

We use one feature and plain gradient descent so the notebook stays CPU-only and deterministic.

In [ ]:
x_train = df["clicks_7d"].to_numpy(dtype=float)
y = df["clicked"].to_numpy(dtype=float)

mean_7d = x_train.mean()
std_7d = x_train.std()
z_train = (x_train - mean_7d) / std_7d

X = np.column_stack([np.ones(n), z_train])
w = np.zeros(2)
learning_rate = 0.15

for step in range(800):
    pred = 1.0 / (1.0 + np.exp(-(X @ w)))
    grad = X.T @ (pred - y) / n
    w = w - learning_rate * grad

print("weights:", np.round(w, 3))

## Score the same requests two ways

The model was trained on `clicks_7d`. Serving now sends `clicks_1d` under the same column name, which is train/serve skew.

In [ ]:
def sigmoid(a):
    return 1.0 / (1.0 + np.exp(-a))

z_offline = (df["clicks_7d"].to_numpy(dtype=float) - mean_7d) / std_7d
z_online_skewed = (df["clicks_1d"].to_numpy(dtype=float) - mean_7d) / std_7d

p_offline = sigmoid(w[0] + w[1] * z_offline)
p_online_skewed = sigmoid(w[0] + w[1] * z_online_skewed)

mean_gap = np.mean(np.abs(p_offline - p_online_skewed))
max_gap = np.max(np.abs(p_offline - p_online_skewed))

print("mean prediction gap:", round(float(mean_gap), 4))
print("max prediction gap:", round(float(max_gap), 4))

## Assert the skew is real

For the same served requests, offline and online predictions should match. They do not, because the offline path used a 7-day window and the online path used a 1-day window.

In [ ]:
assert mean_gap > 0.04
assert max_gap > 0.10

example = int(np.argmax(np.abs(p_offline - p_online_skewed)))

print(df.loc[example, ["clicks_7d", "clicks_1d"]])
print("offline prediction:", round(float(p_offline[example]), 4))
print("skewed online prediction:", round(float(p_online_skewed[example]), 4))

## Repair the contract

If the versioned contract is 7 days, the serving path must retrieve or compute the same 7-day value. Then the score gap disappears for the same model and same requests.

In [ ]:
z_online_fixed = (df["clicks_7d"].to_numpy(dtype=float) - mean_7d) / std_7d
p_online_fixed = sigmoid(w[0] + w[1] * z_online_fixed)

fixed_gap = np.max(np.abs(p_offline - p_online_fixed))

assert fixed_gap == 0.0

print("max gap after unifying definition:", fixed_gap)

## Optional drift signal: PSI

Skew compares offline and online values at the same time. Drift compares populations across time under the same contract. PSI is one compact signal: $\sum_i (a_i-e_i)\log(a_i/e_i)$.

In [ ]:
def psi(expected, actual, bins):
    e_counts = np.histogram(expected, bins=bins)[0].astype(float)
    a_counts = np.histogram(actual, bins=bins)[0].astype(float)
    e_share = np.clip(e_counts / e_counts.sum(), 1e-6, None)
    a_share = np.clip(a_counts / a_counts.sum(), 1e-6, None)
    return float(np.sum((a_share - e_share) * np.log(a_share / e_share)))

next_week_7d = rng.poisson(lam=campaign_quality + 11.0)
bins = np.quantile(clicks_7d, np.linspace(0.0, 1.0, 6))
bins = np.unique(bins)

psi_value = psi(clicks_7d, next_week_7d, bins)

assert psi_value >= 0.0

print("PSI next week versus training week:", round(psi_value, 4))

## Visual check

The left distribution is what the model learned. The skewed serving distribution is much smaller because it is a 1-day count pretending to be a 7-day feature.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))

ax.hist(clicks_7d, bins=30, alpha=0.65, label="offline 7-day")
ax.hist(clicks_1d, bins=30, alpha=0.65, label="online 1-day")
ax.set_title("same feature name, different feature definition")
ax.set_xlabel("click count")
ax.set_ylabel("rows")
ax.legend()

plt.show()